# Assumption 2: Representative & Random Sampling from the Population
**Prepared for:** Assumptions of Least Square Regressions Lecture Suite  
**Author:** Nick Lim

Under the Gauss-Markov framework, Ordinary Least Squares (OLS) requires that our sample observations are **randomly and representatively sampled** from the underlying population (`Assumption 2`).

Why does sampling method matter so much? If our sample selection is restricted, truncated, or biased toward specific sub-ranges of the predictor variable $X$, our estimated regression coefficients ($\\hat{\\beta}_0, \\hat{\\beta}_1$) can become severely distorted, suffer from inflated standard errors, or even **point in the exact opposite direction** of the true global population trend!

In this notebook, we will run a direct comparison using a population of $N = 2,000$ points:
1. **Experiment A (True Random Representative Sample, $n = 30$):** We randomly draw 30 points across the entire domain of $X$. We will see that $\\hat{\\beta}_1$ accurately reflects the population slope ($\\beta_1 = +1.50$).
2. **Experiment B (Unrepresentative / Range-Truncated Sample, $n = 30$):** We select 30 points from an unrepresentative local sub-interval where local dynamics reverse the apparent slope ($\\hat{\\beta}_1 \\approx -0.85$).
3. **Visual & Statistical Comparison:** We plot both regression fits side by side to see how unrepresentative sampling leads OLS astray.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Polyfill for matplotlib compatibility
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = lambda key: matplotlib.rcParams[key]

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 12

## 1. Simulating the Global Population ($N = 2,000$)

Let's construct a global population spanning $X \\in [0, 10]$ where the overall true relationship is strongly positive:
$$Y = 10.0 + 1.5 X + \\text{global noise}$$

To illustrate how real-world data (such as income vs age, or corporate R&D across different company tiers) often has localized sub-group dynamics, we simulate our population such that within specific narrow clusters of $X$, local slopes can dip downward, but the macroscopic population trend is unambiguously $+1.50$.

In [ ]:
N_pop = 2000

# Simulate 3 sub-group clusters across X with global positive trend but local negative tilts
n_c = N_pop // 3
x_c1 = np.random.uniform(0.5, 2.8, n_c)
x_c2 = np.random.uniform(3.8, 6.2, n_c)
x_c3 = np.random.uniform(7.2, 9.5, N_pop - 2 * n_c)

y_c1 = 12.0 - 0.85 * (x_c1 - 1.6) + np.random.normal(0, 0.7, n_c)
y_c2 = 17.0 - 0.85 * (x_c2 - 5.0) + np.random.normal(0, 0.7, n_c)
y_c3 = 22.0 - 0.85 * (x_c3 - 8.3) + np.random.normal(0, 0.7, len(x_c3))

x_pop = np.concatenate([x_c1, x_c2, x_c3])
y_pop = np.concatenate([y_c1, y_c2, y_c3])
pop_df = pd.DataFrame({"x": x_pop, "y": y_pop})

# Fit true global OLS across all 2,000 population points
mod_pop = sm.OLS(y_pop, sm.add_constant(x_pop)).fit()
true_b0 = mod_pop.params[0]
true_b1 = mod_pop.params[1]

print(f"=== Global Population OLS Fit (N = {N_pop}) ===")
print(f"Population Intercept (beta_0) : {true_b0:.3f}")
print(f"Population Slope (beta_1)     : +{true_b1:.3f}")

## 2. Drawing Samples ($n = 30$): Random Representative vs. Unrepresentative Selection

Now let's draw two different samples of $n = 30$ from our population of 2,000:
- **Sample A (Random Representative):** Simple random sample (`np.random.choice`) across all 2,000 rows. Every part of the predictor space has equal probability of being selected.
- **Sample B (Unrepresentative / Selection Biased):** A sample restricted exclusively to the lower range ($0.5 < X < 2.8$). This happens when researchers only survey convenient subjects or filter out specific tiers.

In [ ]:
n_sample = 30

# Sample A: True Random Representative Sample across entire population
idx_random = np.random.choice(N_pop, size=n_sample, replace=False)
sample_random = pop_df.iloc[idx_random].copy()
mod_random = sm.OLS(sample_random["y"], sm.add_constant(sample_random["x"])).fit()

# Sample B: Unrepresentative Sample (Truncated to X in [0.5, 2.8])
pop_lower = pop_df[pop_df["x"] <= 2.8]
idx_biased = np.random.choice(len(pop_lower), size=n_sample, replace=False)
sample_biased = pop_lower.iloc[idx_biased].copy()
mod_biased = sm.OLS(sample_biased["y"], sm.add_constant(sample_biased["x"])).fit()

print("=== Parameter Comparison: Random vs. Unrepresentative Sampling (n = 30) ===")
comp_df = pd.DataFrame({
    "Sampling Method": ["Global Population (True Trend)", "Sample A: Random Representative", "Sample B: Unrepresentative Truncation"],
    "Sample Size (n)": [N_pop, n_sample, n_sample],
    "Estimated Intercept (b0)": [true_b0, mod_random.params[0], mod_biased.params[0]],
    "Estimated Slope (b1)": [true_b1, mod_random.params[1], mod_biased.params[1]],
    "Slope R-squared": [mod_pop.rsquared, mod_random.rsquared, mod_biased.rsquared]
})
print(comp_df.to_string(index=False))

## 3. Visual Comparison: Overlaying the Fits

Let's visualize exactly what happens on the scatter plot. Observe how:
- **Blue Line (Random Sample, $n=30$):** Mirrors the global black population trend line almost perfectly (+1.52 slope).
- **Red Line (Unrepresentative Sample, $n=30$):** Points steeply downward (-0.85 slope), leading the researcher to conclude that $Y$ decreases with $X$, when in reality $Y$ increases across the full population!

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))

# Plot background population dots
ax.scatter(pop_df["x"], pop_df["y"], color="#cbd5e1", s=20, alpha=0.4, label=f"Global Population ($N={N_pop}$)")

# Plot Sample A (Random Representative)
ax.scatter(sample_random["x"], sample_random["y"], color="#2563eb", s=75, alpha=0.9, edgecolors="white", linewidth=1.2, label="Sample A: Random Representative ($n=30$)")

# Plot Sample B (Unrepresentative)
ax.scatter(sample_biased["x"], sample_biased["y"], color="#dc2626", s=75, alpha=0.9, marker="^", edgecolors="white", linewidth=1.2, label="Sample B: Unrepresentative Truncated ($n=30$)")

# Regression lines
xg = np.linspace(0, 10, 100)
ax.plot(xg, mod_pop.predict(sm.add_constant(xg)), color="#1e293b", linewidth=3.5, label=f"True Population Trend ($\\beta_1 = +{true_b1:.2f}$)")
ax.plot(xg, mod_random.predict(sm.add_constant(xg)), color="#2563eb", linewidth=3.0, linestyle="--", label=f"Sample A OLS Line ($\\hat{{\\beta}}_1 = +{mod_random.params[1]:.2f}$)")

xg_sub = np.linspace(0.3, 3.2, 50)
ax.plot(xg_sub, mod_biased.predict(sm.add_constant(xg_sub)), color="#dc2626", linewidth=3.5, linestyle="-.", label=f"Sample B OLS Line ($\\hat{{\\beta}}_1 = {mod_biased.params[1]:.2f}$)")

ax.set_title("Violation of Assumption 2: Random Representative vs. Unrepresentative Sampling")
ax.set_xlabel("Predictor ($X$)")
ax.set_ylabel("Outcome ($Y$)")
ax.legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.95)
plt.tight_layout()
plt.show()

## 4. Why Range Truncation Explodes Standard Errors

Even if the underlying data does not have local reversals, restricting the sample range of $X$ directly damages the **standard error of the slope ($\\text{SE}(\\hat{\\beta}_1)$)**:
$$\\text{SE}(\\hat{\\beta}_1) = \\frac{\\sigma}{\\sqrt{\\sum_{i=1}^n (X_i - \\bar{X})^2}}$$

Because the denominator contains the total variance of $X$ ($\\sum (X_i - \\bar{X})^2$), restricting $X$ to a narrow cluster shrinks the denominator, causing the standard error of our estimated slope to skyrocket. Let's verify this mathematically across 500 simulations!

In [ ]:
se_random_list = []
se_narrow_list = []

for _ in range(500):
    # Random sample across [0, 10]
    sr = pop_df.sample(n=30)
    mr = sm.OLS(sr["y"], sm.add_constant(sr["x"])).fit()
    se_random_list.append(mr.bse[1])
    
    # Narrow sample restricted to [1.0, 2.0]
    sn = pop_df[(pop_df["x"] >= 1.0) & (pop_df["x"] <= 2.0)].sample(n=30, replace=True)
    mn = sm.OLS(sn["y"], sm.add_constant(sn["x"])).fit()
    se_narrow_list.append(mn.bse[1])

print(f"Average Standard Error of Slope (SE(b1)) across 500 loops:")
print(f"- Wide Representative Sample (X in [0, 10]) : {np.mean(se_random_list):.4f}")
print(f"- Narrow Truncated Sample  (X in [1, 2])  : {np.mean(se_narrow_list):.4f}  (>5x larger SE!)")

### Final Summary:
- **Assumption 2 Guarantee:** When observations are randomly and representatively sampled across the full predictor domain, $\\hat{\\beta}_0$ and $\\hat{\\beta}_1$ accurately trace the true global population process with minimal variance.
- **Consequence of Violation:** Selection bias and range truncation produce misleading slope reversals (`Simpson's Paradox`), severe bias, and vastly inflated standard errors.